# ModernBERT Spoiler Detection

This notebook fine-tunes ModernBERT for spoiler detection on IMDb movie reviews and Goodreads book reviews.

We evaluate the model in four settings:

1. IMDb → IMDb (in-domain)
2. IMDb → Goodreads (cross-domain)
3. Goodreads → Goodreads (in-domain)
4. Goodreads → IMDb (cross-domain)

The same train, validation, and test splits as in the TF-IDF baseline are used to ensure a direct comparison between both approaches.


## Setup


In [1]:
!git clone -b dev https://github.com/rafaelTamm/NLE-Project-Spoiler-Detection.git
%cd NLE-Project-Spoiler-Detection


Cloning into 'NLE-Project-Spoiler-Detection'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 70 (delta 27), reused 20 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 31.46 MiB | 24.80 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/NLE-Project-Spoiler-Detection


In [2]:
!pip install -q datasets transformers accelerate


In [3]:
import os
import pandas as pd
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## Load Data


In [4]:
imdb_train = pd.read_csv("data/imdb_train.csv")
imdb_val = pd.read_csv("data/imdb_val.csv")
imdb_test = pd.read_csv("data/imdb_test.csv")

goodreads_train = pd.read_csv("data/goodreads_train.csv")
goodreads_val = pd.read_csv("data/goodreads_val.csv")
goodreads_test = pd.read_csv("data/goodreads_test.csv")

datasets = {
    "IMDb train": imdb_train,
    "IMDb val": imdb_val,
    "IMDb test": imdb_test,
    "Goodreads train": goodreads_train,
    "Goodreads val": goodreads_val,
    "Goodreads test": goodreads_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)} reviews")


IMDb train: 13913 reviews
IMDb val: 2821 reviews
IMDb test: 3266 reviews
Goodreads train: 13982 reviews
Goodreads val: 2934 reviews
Goodreads test: 3084 reviews


## Prepare Hugging Face Datasets


In [5]:
imdb_dataset = DatasetDict({
    "train": Dataset.from_pandas(imdb_train),
    "validation": Dataset.from_pandas(imdb_val),
    "test": Dataset.from_pandas(imdb_test)
})

goodreads_dataset = DatasetDict({
    "train": Dataset.from_pandas(goodreads_train),
    "validation": Dataset.from_pandas(goodreads_val),
    "test": Dataset.from_pandas(goodreads_test)
})


## Prepare Labels


In [6]:
def prepare_labels(example):
    example["label"] = int(example["is_spoiler"])
    return example

imdb_dataset = imdb_dataset.map(prepare_labels)
goodreads_dataset = goodreads_dataset.map(prepare_labels)

print("IMDb:", imdb_dataset["train"][0]["is_spoiler"], "->", imdb_dataset["train"][0]["label"])
print("Goodreads:", goodreads_dataset["train"][0]["is_spoiler"], "->", goodreads_dataset["train"][0]["label"])


Map:   0%|          | 0/13913 [00:00<?, ? examples/s]

Map:   0%|          | 0/2821 [00:00<?, ? examples/s]

Map:   0%|          | 0/3266 [00:00<?, ? examples/s]

Map:   0%|          | 0/13982 [00:00<?, ? examples/s]

Map:   0%|          | 0/2934 [00:00<?, ? examples/s]

Map:   0%|          | 0/3084 [00:00<?, ? examples/s]

IMDb: False -> 0
Goodreads: False -> 0


## ModernBERT Tokenizer

The previous length analysis showed that a maximum sequence length of 1024 preserves substantially more spoiler-review content than 512 while keeping training feasible.


In [7]:
checkpoint = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

MAX_LENGTH = 1024


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

## Tokenization


In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["review_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_imdb = imdb_dataset.map(tokenize_function, batched=True)
tokenized_goodreads = goodreads_dataset.map(tokenize_function, batched=True)

print(tokenized_imdb["train"][0].keys())
print("IMDb token count:", len(tokenized_imdb["train"][0]["input_ids"]))
print("Goodreads token count:", len(tokenized_goodreads["train"][0]["input_ids"]))


Map:   0%|          | 0/13913 [00:00<?, ? examples/s]

Map:   0%|          | 0/2821 [00:00<?, ? examples/s]

Map:   0%|          | 0/3266 [00:00<?, ? examples/s]

Map:   0%|          | 0/13982 [00:00<?, ? examples/s]

Map:   0%|          | 0/2934 [00:00<?, ? examples/s]

Map:   0%|          | 0/3084 [00:00<?, ? examples/s]

dict_keys(['review_text', 'is_spoiler', 'item_id', 'label', 'input_ids', 'attention_mask'])
IMDb token count: 138
Goodreads token count: 71


## Dynamic Padding


In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## Evaluation Metrics


In [10]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0)
    }


## GPU Check


In [11]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available.")


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Google Drive Checkpoints


In [12]:
from google.colab import drive
drive.mount("/content/drive")

IMDB_OUTPUT_DIR = "/content/drive/MyDrive/NLP_Spoiler_Project/modernbert_imdb"
os.makedirs(IMDB_OUTPUT_DIR, exist_ok=True)

print("Checkpoint directory:", IMDB_OUTPUT_DIR)
print("Exists:", os.path.exists(IMDB_OUTPUT_DIR))


Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/NLP_Spoiler_Project/modernbert_imdb
Exists: True


## IMDb Model


In [ ]:
imdb_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## IMDb Training Setup


In [ ]:
imdb_training_args = TrainingArguments(
    output_dir=IMDB_OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    report_to="none"
)

imdb_trainer = Trainer(
    model=imdb_model,
    args=imdb_training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


## IMDb Fine-Tuning

**Stop here until the runtime/GPU is ready for the final training run.**

The test sets are not used for model selection.


In [ ]:
# Sanity check: no movie/book appears in more than one split

def check_split_overlap(train_df, val_df, test_df, name):
    train_ids = set(train_df["item_id"])
    val_ids = set(val_df["item_id"])
    test_ids = set(test_df["item_id"])

    print(name)
    print("Train / Validation overlap:", len(train_ids & val_ids))
    print("Train / Test overlap:", len(train_ids & test_ids))
    print("Validation / Test overlap:", len(val_ids & test_ids))


check_split_overlap(imdb_train, imdb_val, imdb_test, "IMDb")
check_split_overlap(
    goodreads_train,
    goodreads_val,
    goodreads_test,
    "Goodreads"
)

IMDb
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0
Goodreads
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0


In [ ]:
imdb_train_result = imdb_trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,2.000170,0.559371,0.713577,0.781060,0.617869,0.689946,0.711903
2,1.944008,0.619562,0.714286,0.746393,0.675601,0.709235,0.714199
3,1.113793,1.474191,0.703651,0.720599,0.694845,0.707488,0.703600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## IMDb Evaluation


In [ ]:
imdb_in_domain = imdb_trainer.evaluate(tokenized_imdb["test"])
imdb_to_goodreads = imdb_trainer.evaluate(tokenized_goodreads["test"])

print("IMDb → IMDb:", imdb_in_domain)
print("IMDb → Goodreads:", imdb_to_goodreads)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
1.113793,0.576398,3,0.731782,0.743252,0.699071,0.720485,0.731343


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
1.113793,0.654532,3,0.731193,0.694887,0.821847,0.753053,0.729070


IMDb → IMDb: {'eval_loss': 0.5763980150222778, 'eval_accuracy': 0.7317819963257808, 'eval_precision': 0.7432521395655036, 'eval_recall': 0.6990712074303406, 'eval_f1': 0.7204850031908104, 'eval_macro_f1': 0.7313431490350757}
IMDb → Goodreads: {'eval_loss': 0.6545320153236389, 'eval_accuracy': 0.7311932555123216, 'eval_precision': 0.6948873007146784, 'eval_recall': 0.8218465539661899, 'eval_f1': 0.7530533214179327, 'eval_macro_f1': 0.7290702395065473}


## Goodreads Fine-Tuning

A fresh ModernBERT model is fine-tuned on the Goodreads training split.
The Goodreads validation split is used for model selection.

The selected model is evaluated on:
- Goodreads test (in-domain)
- IMDb test (cross-domain)

In [16]:
GOODREADS_OUTPUT_DIR = "/content/drive/MyDrive/NLP_Spoiler_Project/modernbert_goodreads"

os.makedirs(GOODREADS_OUTPUT_DIR, exist_ok=True)

print("Goodreads checkpoint directory:", GOODREADS_OUTPUT_DIR)
print("Exists:", os.path.exists(GOODREADS_OUTPUT_DIR))

Goodreads checkpoint directory: /content/drive/MyDrive/NLP_Spoiler_Project/modernbert_goodreads
Exists: True


In [17]:
goodreads_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
goodreads_training_args = TrainingArguments(
    output_dir=GOODREADS_OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    report_to="none"
)

In [19]:
goodreads_trainer = Trainer(
    model=goodreads_model,
    args=goodreads_training_args,
    train_dataset=tokenized_goodreads["train"],
    eval_dataset=tokenized_goodreads["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [20]:
goodreads_train_result = goodreads_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,1.555735,0.367688,0.847307,0.815691,0.896033,0.853977,0.846988
2,1.139061,0.474401,0.855147,0.844061,0.870041,0.856854,0.855126
3,0.442940,1.010789,0.851738,0.838944,0.869357,0.853880,0.851706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Goodreads Evaluation

The selected Goodreads model is evaluated on:
- Goodreads test (in-domain)
- IMDb test (cross-domain)

In [21]:
goodreads_in_domain = goodreads_trainer.evaluate(
    tokenized_goodreads["test"]
)

goodreads_to_imdb = goodreads_trainer.evaluate(
    tokenized_imdb["test"]
)

print("Goodreads → Goodreads:")
print(goodreads_in_domain)

print("\nGoodreads → IMDb:")
print(goodreads_to_imdb)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
0.442940,0.496986,3,0.852789,0.851036,0.854356,0.852693,0.852789


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
0.442940,1.205881,3,0.663503,0.691111,0.577709,0.629342,0.660620


Goodreads → Goodreads:
{'eval_loss': 0.49698612093925476, 'eval_accuracy': 0.8527885862516212, 'eval_precision': 0.8510362694300518, 'eval_recall': 0.8543563068920677, 'eval_f1': 0.8526930564568462, 'eval_macro_f1': 0.8527885243398943}

Goodreads → IMDb:
{'eval_loss': 1.2058813571929932, 'eval_accuracy': 0.6635027556644213, 'eval_precision': 0.6911111111111111, 'eval_recall': 0.5777089783281734, 'eval_f1': 0.6293423271500843, 'eval_macro_f1': 0.6606201403061888}


## ModernBERT Results

The following table summarizes the in-domain and cross-domain evaluation results for both fine-tuned ModernBERT models.

In [23]:
modernbert_results = pd.DataFrame([
    {
        "Experiment": "IMDb → IMDb",
        "Accuracy": 0.7317819963257808,
        "Precision": 0.7432521395655036,
        "Recall": 0.6990712074303406,
        "F1": 0.7204850031908104,
        "Macro F1": 0.7313431490350757,
    },
    {
        "Experiment": "IMDb → Goodreads",
        "Accuracy": 0.7311932555123216,
        "Precision": 0.6948873007146784,
        "Recall": 0.8218465539661899,
        "F1": 0.7530533214179327,
        "Macro F1": 0.7290702395065473,
    },
    {
        "Experiment": "Goodreads → Goodreads",
        "Accuracy": goodreads_in_domain["eval_accuracy"],
        "Precision": goodreads_in_domain["eval_precision"],
        "Recall": goodreads_in_domain["eval_recall"],
        "F1": goodreads_in_domain["eval_f1"],
        "Macro F1": goodreads_in_domain["eval_macro_f1"],
    },
    {
        "Experiment": "Goodreads → IMDb",
        "Accuracy": goodreads_to_imdb["eval_accuracy"],
        "Precision": goodreads_to_imdb["eval_precision"],
        "Recall": goodreads_to_imdb["eval_recall"],
        "F1": goodreads_to_imdb["eval_f1"],
        "Macro F1": goodreads_to_imdb["eval_macro_f1"],
    },
])

modernbert_results.round(4)

,Experiment,Accuracy,Precision,Recall,F1,Macro F1
0,IMDb → IMDb,0.7318,0.7433,0.6991,0.7205,0.7313
1,IMDb → Goodreads,0.7312,0.6949,0.8218,0.7531,0.7291
2,Goodreads → Goodreads,0.8528,0.8510,0.8544,0.8527,0.8528
3,Goodreads → IMDb,0.6635,0.6911,0.5777,0.6293,0.6606


In [25]:
imdb_to_goodreads_drop = (
    modernbert_results.loc[0, "Macro F1"]
    - modernbert_results.loc[1, "Macro F1"]
)

goodreads_to_imdb_drop = (
    modernbert_results.loc[2, "Macro F1"]
    - modernbert_results.loc[3, "Macro F1"]
)

print(f"IMDb → Goodreads Macro F1 drop: {imdb_to_goodreads_drop:.4f}")
print(f"Goodreads → IMDb Macro F1 drop: {goodreads_to_imdb_drop:.4f}")

IMDb → Goodreads Macro F1 drop: 0.0023
Goodreads → IMDb Macro F1 drop: 0.1922
